# StaticSpinSucep: spin susceptibility from excitonic BSE

This notebook separates the spin-susceptibility problem from the excitonic Bethe-Salpeter notebook. The goal is to build first, on a fixed magnetic background, two contributions:

$$R^R = R^R_{\rm bare} + R^R_{1D},$$

where $R_{\rm bare}$ is the electronic spin-spin bubble and $R_{1D}$ is the contribution from the excitonic action that contains a single excitonic propagator $D$.


## Objects to compute

The retarded spin susceptibility is

$$R^{ab,R}_{ll'}(t)=-i\Theta(t)\langle [s_l^a(t),s_{l'}^b(0)]\rangle,$$

with $l,l'=1,2$ layers and $a,b=x,y,z$. We use

$$s_l^a=\frac{1}{2}c_l^\dagger\sigma^a c_l.$$

For a Hamiltonian without inter-layer tunneling, the expected benchmark is that $R_{\rm bare}$ is diagonal in layer. The excitonic contribution enters because

$$D_M^{-1}=V^{-1}-\Pi[M],$$

and therefore the expansion of $i\mathrm{Tr}\ln D_M^{-1}$ generates, to second order in fluctuations of $M$,

$$R^{ab,R}_{1D,ll'}\sim -i\,\mathrm{Tr}\left[D^R[M]\,\frac{\partial^2\Pi^R[M]}{\partial M_l^a\partial M_{l'}^b}\right].$$

The term with two propagators, $D(\partial_M\Pi)D(\partial_M\Pi)$, is left for later.


In [ ]:
using LinearAlgebra
using Printf
using PyPlot

rc("font", family="serif")
rc("font", size=13)
rc("axes", labelsize=14, titlesize=14)
rc("legend", fontsize=10);


In [ ]:
# -----------------------------
# Physical and numerical parameters
# -----------------------------
L = 100
γ = 1.0
Δ_layer = 8.0
Jsd = 0.35
Mmag = 1.0
θ = 0.0
V = -1.2
T = 0.08

# Chemical potentials diagonal in the Hamiltonian.
μchem = 0.0
μbias = 0.0
μ1 = μchem + μbias/2
μ2 = μchem - μbias/2

ηΩ = 0.16
Ωmin, Ωmax, NΩ = 0.0, 15.0, 401

ks = collect(range(-π, stop=π - 2π/L, length=L))
Ωs = collect(range(Ωmin, Ωmax, length=NΩ));


In [ ]:
# Pauli matrices in physical spin space.
σ0 = ComplexF64[1 0; 0 1]
σx = ComplexF64[0 1; 1 0]
σy = ComplexF64[0 -1im; 1im 0]
σz = ComplexF64[1 0; 0 -1]

# Physical spin vertices s^a = sigma^a/2.
S_spin = [σx, σy, σz] ./ 2
spin_labels = ["x", "y", "z"]

# Excitonic vertices Gamma^mu = sigma^mu/sqrt(2), mu=0,x,y,z.
Γ_exciton = [σ0, σx, σy, σz] ./ sqrt(2)
channel_labels = ["0", "x", "y", "z"];


In [ ]:
ϵk(k; γ=γ) = -2γ*cos(k)

function fermi(E; T=T, μ=0.0)
    x = (E - μ) / T
    x > 60 && return 0.0
    x < -60 && return 1.0
    return 1 / (exp(x) + 1)
end

layer_chemical_potential(layer::Int) = layer == 1 ? μ1 : μ2

function h_layer(k, M; layer::Int)
    offset = layer == 1 ? +Δ_layer/2 : Δ_layer/2
    energy = (ϵk(k) + offset) * σ0 - Jsd * (M[1]*σx + M[2]*σy + M[3]*σz)
    energy .*= (-1)^(layer+1)
    energy .-= layer_chemical_potential(layer) * σ0
    return energy
end

function magnetizations(θ)
    M1 = Mmag .* [0.0, 0.0, 1.0]
    M2 = Mmag .* [sin(θ), 0.0, cos(θ)]
    return M1, M2
end

function diagonalize_layers(ks, θ)
    M1, M2 = magnetizations(θ)
    E1 = zeros(Float64, 2, length(ks))
    E2 = zeros(Float64, 2, length(ks))
    U1 = Vector{Matrix{ComplexF64}}(undef, length(ks))
    U2 = Vector{Matrix{ComplexF64}}(undef, length(ks))

    for (ik, k) in enumerate(ks)
        F1 = eigen(Hermitian(h_layer(k, M1; layer=1)))
        F2 = eigen(Hermitian(h_layer(k, M2; layer=2)))
        E1[:, ik] .= F1.values
        E2[:, ik] .= F2.values
        U1[ik] = Matrix{ComplexF64}(F1.vectors)
        U2[ik] = Matrix{ComplexF64}(F2.vectors)
    end

    return (; M1, M2, E1, E2, U1, U2)
end;


## $R_{\rm bare}$: electronic spin-spin bubble

In equilibrium we use the retarded band expression

$$R^{ab,R}_{l,{\rm bare}}(\Omega)=\frac{1}{L}\sum_{k,n,m}\frac{f(E_{lnk})-f(E_{lmk})}{\Omega+i\eta-(E_{lmk}-E_{lnk})}\,S^a_{nm}(k)S^b_{mn}(k).$$

Because there is no inter-layer tunneling, this contribution is diagonal in layer: $R_{12,{\rm bare}}=R_{21,{\rm bare}}=0$.


In [ ]:
function spin_susceptibility_bare_band(Ωs, ks, θ; η=ηΩ)
    data = diagonalize_layers(ks, θ)
    R = zeros(ComplexF64, 2, 2, 3, 3, length(Ωs))

    for layer in 1:2
        E = layer == 1 ? data.E1 : data.E2
        U = layer == 1 ? data.U1 : data.U2

        for (ik, _) in enumerate(ks)
            Uk = U[ik]
            Sa = [Uk' * S_spin[a] * Uk for a in 1:3]

            for n in 1:2, m in 1:2
                ΔE = E[m, ik] - E[n, ik]
                occ = fermi(E[n, ik]) - fermi(E[m, ik])

                for (iΩ, Ω) in enumerate(Ωs)
                    pref = occ / (Ω + 1im*η - ΔE) / length(ks)
                    for a in 1:3, b in 1:3
                        R[layer, layer, a, b, iΩ] += pref * Sa[a][n, m] * Sa[b][m, n]
                    end
                end
            end
        end
    end

    return R, data
end;


## $R_{1D}$: excitonic term with a single $D$

The next contribution comes from

$$i\mathrm{Tr}\ln D_M^{-1},\qquad D_M^{-1}=V^{-1}-\Pi[M].$$

The part containing a single excitonic propagator is

$$R^{ab,R}_{1D,ll'}\sim -i\,\mathrm{Tr}_{\mu\nu,\Omega}\left[D^R_{\mu\nu}(\Omega;M)\,\partial_{M_l^a}\partial_{M_{l'}^b}\Pi^R_{\nu\mu}(\Omega;M)\right].$$

Before implementing it, we need to fix the sign convention, the factor of $J_{sd}$ from the variation with respect to $M$, and whether we want a static version or one with external-frequency dependence.


In [ ]:
# Optional smoke plot: bare response of layer 1 for the fixed angle.
# Rbare, layer_data = spin_susceptibility_bare_band(Ωs, ks, θ)
# fig, ax = subplots(figsize=(6.8, 4.4))
# for a in 1:3
#     A = [-2 * imag(Rbare[1, 1, a, a, iΩ]) for iΩ in axes(Rbare, 5)]
#     ax.plot(Ωs, A, lw=1.6, label="R_" * spin_labels[a] * spin_labels[a])
# end
# ax.set_xlabel("Omega")
# ax.set_ylabel("-2 Im R_bare")
# ax.set_title("Bare spin susceptibility, layer 1")
# ax.legend(frameon=false)
# ax.grid(alpha=0.18)
# fig.tight_layout()
